This Notebook tries to provide an explicit approach to implementing Retrieval-Augmented Generation (RAG) to a given LLM.

My approach consists in mounting my Google Drive unit and then scraping a fixed folder to obtain the PDFs from there.

I will try to keep the main variables outside the function definitions in order to be changed and tuned as necessary.

## 1. Extract PDF


Mount the Google Drive from the account that is executing the Notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The `folder_path` can be any folder with PDFs in it.

In [ ]:
import os

folder_path = '/content/drive/MyDrive/RAG'

# Filter to only include files that end with .pdf
files = [f for f in os.listdir(folder_path) if f.lower().endswith('.pdf')]

for f in files:
    print(f)

manual_servicio.pdf
CV_LOGU_AI_E (9).pdf


In [ ]:
pip install PyPDF2

In [ ]:
import PyPDF2

for file in files:
  print(file)

manual_servicio.pdf
CV_LOGU_AI_E (9).pdf


In [ ]:
import PyPDF2
import os

# Direct string to store all combined content
texts = ""

for file in files:
    full_path = os.path.join(folder_path, file)

    with open(full_path, 'rb') as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        for page in pdf_reader.pages:
            text = page.extract_text()
            texts += text + "\n" # Append directly to the string

    print(f"Extracted: {file}")

Extracted: manual_servicio.pdf
Extracted: CV_LOGU_AI_E (9).pdf


In [ ]:
print(texts)

¡Las ofertas continúan! Adelántate a Navidad Aprovecha🏷 🎄
Detalle del pedido
Se entregó en tu domicilioMENÚ
Envío a domicilioBusca tus productos
Tarjeta de débito / créditoNúmero de pedido: 155718890
Fecha de compra: 8 de noviembre de 2024
Ver detalles de pagoPagaste
Envío
COPPEL EXPRESSConﬁrmandoElige tuciudad de entrega
Preparando En caminoEntregadoMi Cuenta Carrito
Detalles de tu pago
$12,136
Envío
COPPEL EXPRESSConﬁrmando 
Vendido por Coppel
 x 1 Unidad
 PreparandoRecibe
LUIS OCTAVIO GARCIA
Domicilio
MARIA CURIE 308 MONTERREY NUEVO LEON 64700
Recibe
LUIS OCTAVIO GARCIA
Domicilio
MARIA CURIE 308 MONTERREY NUEVO LEON 64700En caminoEntregadoTus productos
Envío a domicilioSmartphone Google Pixel 8 5g De 256 Gb Obsidian - Desbloqueado
$12,136
Se entregó en tu domicilioGenerar factura
LUIS OCTAVIO GARCIA URIBE
AI Solutions Engineer | M. Sc. in Mathematics
luisoctavio98@gmail.com/ne+52 667 304 9739Monterrey, N.L., México
Solutions Engineer with hands-on delivery of AI/LLM projects in
Pyth

## 2. Vectorize extracted text

In [ ]:
pip install nltk # Natural Language Toolkit

In [ ]:
import nltk
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# In my files it's unclear when a sentence begins or ends, so I will tokenize via number of lines.
lines = texts.split("\n")

In [ ]:
# Sanity check to verify number of lines obtained.
print(len(lines))
print(lines)

104
['¡Las ofertas continúan! Adelántate a Navidad Aprovecha🏷 🎄', 'Detalle del pedido', 'Se entregó en tu domicilioMENÚ', 'Envío a domicilioBusca tus productos', 'Tarjeta de débito / créditoNúmero de pedido: 155718890', 'Fecha de compra: 8 de noviembre de 2024', 'Ver detalles de pagoPagaste', 'Envío', 'COPPEL EXPRESSConﬁrmandoElige tuciudad de entrega', 'Preparando En caminoEntregadoMi Cuenta Carrito', 'Detalles de tu pago', '$12,136', 'Envío', 'COPPEL EXPRESSConﬁrmando ', 'Vendido por Coppel', ' x 1 Unidad', ' PreparandoRecibe', 'LUIS OCTAVIO GARCIA', 'Domicilio', 'MARIA CURIE 308 MONTERREY NUEVO LEON 64700', 'Recibe', 'LUIS OCTAVIO GARCIA', 'Domicilio', 'MARIA CURIE 308 MONTERREY NUEVO LEON 64700En caminoEntregadoTus productos', 'Envío a domicilioSmartphone Google Pixel 8 5g De 256 Gb Obsidian - Desbloqueado', '$12,136', 'Se entregó en tu domicilioGenerar factura', 'LUIS OCTAVIO GARCIA URIBE', 'AI Solutions Engineer | M. Sc. in Mathematics', 'luisoctavio98@gmail.com/ne+52 667 304 973

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


We remove the stepwords from the extracted sentences.

In [ ]:
stops = set(stopwords.words('spanish')).union(set(stopwords.words('english')))

In [ ]:
clean_lines = []
for line in lines:
    words = word_tokenize(line)
    clean_words = [p for p in words if p.lower() not in stops]
    clean_line = " ".join(clean_words)
    clean_lines.append(clean_line)

print(clean_lines)

['¡Las ofertas continúan ! Adelántate Navidad Aprovecha🏷 🎄', 'Detalle pedido', 'entregó domicilioMENÚ', 'Envío domicilioBusca productos', 'Tarjeta débito / créditoNúmero pedido : 155718890', 'Fecha compra : 8 noviembre 2024', 'Ver detalles pagoPagaste', 'Envío', 'COPPEL EXPRESSConﬁrmandoElige tuciudad entrega', 'Preparando caminoEntregadoMi Cuenta Carrito', 'Detalles pago', '$ 12,136', 'Envío', 'COPPEL EXPRESSConﬁrmando', 'Vendido Coppel', 'x 1 Unidad', 'PreparandoRecibe', 'LUIS OCTAVIO GARCIA', 'Domicilio', 'MARIA CURIE 308 MONTERREY NUEVO LEON 64700', 'Recibe', 'LUIS OCTAVIO GARCIA', 'Domicilio', 'MARIA CURIE 308 MONTERREY NUEVO LEON 64700En caminoEntregadoTus productos', 'Envío domicilioSmartphone Google Pixel 8 5g 256 Gb Obsidian - Desbloqueado', '$ 12,136', 'entregó domicilioGenerar factura', 'LUIS OCTAVIO GARCIA URIBE', 'AI Solutions Engineer | M. Sc . Mathematics', 'luisoctavio98 @ gmail.com/ne+52 667 304 9739Monterrey , N.L. , México', 'Solutions Engineer hands-on delivery AI/L

Now we create chunks where `size` is the number of sentences in a chunk.

In [ ]:
def create_chunks(clean_lines, size=10):
    chunks = []
    for i in range(0, len(lines), size):
        chunk = " ".join(lines[i:i+size])
        chunks.append(chunk)
    return chunks

In [ ]:
chunks = create_chunks(clean_lines)
print(len(chunks))

11


Finally we vectorize our chunks.

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(chunks)

In [ ]:
print(vectors.shape)


(11, 325)


To analyze the vectorized data we can retrieve it using `pandas` and `numpy`, but that is completely optional.

In [ ]:
import pandas as pd

# Obtain a list with all the different words.
words = vectorizer.get_feature_names_out()
print(len(words))
print(words)

325
['00' '10' '12' '136' '15' '155718890' '2018' '2021méxico' '2022sinaloa'
 '2024' '2024guanajuato' '2024remote' '2025' '2025remote' '2026remote'
 '256' '304' '308' '52' '5g' '64700' '64700en' '667' '80' '9739monterrey'
 'ability' 'adelántate' 'advanced' 'ai' 'algebra' 'algebras' 'algorithms'
 'align' 'analysis' 'analytical' 'analyzed' 'and' 'api' 'apis' 'aprovecha'
 'at' 'august' 'auto' 'automation' 'autónoma' 'aws' 'bachelor' 'banach'
 'behavior' 'bi' 'building' 'built' 'by' 'c1' 'calculus' 'calibration'
 'call' 'calls' 'caminoentregadomi' 'caminoentregadotus' 'carrito'
 'centro' 'checks' 'cimat' 'classiﬁcation' 'cleaning' 'client' 'com'
 'complex' 'complexity' 'compra' 'computational' 'consistency' 'continúan'
 'coppel' 'corrected' 'created' 'créditonúmero' 'cuenta' 'curie'
 'customized' 'dashboards' 'data' 'datasets' 'de' 'debugging' 'deepeval'
 'del' 'delivery' 'desbloqueado' 'design' 'designed' 'detalle' 'detalles'
 'developed' 'distinction' 'diﬀerent' 'diﬀerential' 'diﬃculty' 

In [ ]:
import numpy as np

matrix = vectors.toarray()
print(matrix[0])
# This shows with non-zero the words present in the chunk, and their normalized relevance.

[0.         0.         0.         0.         0.         0.13460207
 0.         0.         0.         0.10118263 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.13460207 0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.13460207 0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.13460207 0.
 0.13460207 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.13460207 0.
 0.         0.13460207 0.11505295 0.         0.         0.13460207
 0.13460207 0.         0.         0.         0.         0.
 0.51940893 0.         0.         0.13460207 0.         0.
 0.         0.         0.13460207 0.11505295 0.         0.
 0.         0.         0.         0.    

See the relevant words in each chunk

In [ ]:
def explorar_vectores(vectorizer, vectors, chunks):
    words = vectorizer.get_feature_names_out()
    matrix = vectors.toarray()

    print(f"Total chunks: {matrix.shape[0]}")
    print(f"Total unique words: {matrix.shape[1]}")

    for i, chunk in enumerate(chunks):
        print(f"\n CHUNK {i}: {chunk[:100]}...")  # first 100 characters
        print("Relevant words:")

        # Sort by relevance
        index = np.argsort(matrix[i])[::-1]

        for idx in index[:5]:  # top 5 words
            if matrix[i][idx] > 0:
                print(f"  - {words[idx]}: {matrix[i][idx]:.3f}")

explorar_vectores(vectorizer, vectors, chunks)

Total chunks: 11
Total unique words: 325

 CHUNK 0: ¡Las ofertas continúan! Adelántate a Navidad Aprovecha🏷 🎄 Detalle del pedido Se entregó en tu domici...
Relevant words:
  - de: 0.519
  - pedido: 0.269
  - en: 0.202
  - envío: 0.202
  - compra: 0.135

 CHUNK 1: Detalles de tu pago $12,136 Envío COPPEL EXPRESSConﬁrmando  Vendido por Coppel  x 1 Unidad  Preparan...
Relevant words:
  - coppel: 0.374
  - unidad: 0.219
  - pago: 0.219
  - por: 0.219
  - vendido: 0.219

 CHUNK 2: Recibe LUIS OCTAVIO GARCIA Domicilio MARIA CURIE 308 MONTERREY NUEVO LEON 64700En caminoEntregadoTus...
Relevant words:
  - luis: 0.254
  - garcia: 0.254
  - octavio: 0.254
  - 9739monterrey: 0.149
  - 64700en: 0.149

 CHUNK 3: Solutions Engineer with hands-on delivery of AI/LLM projects in Python, building LLM-powered automat...
Relevant words:
  - llm: 0.320
  - automation: 0.284
  - skills: 0.284
  - python: 0.242
  - and: 0.214

 CHUNK 4: Sklearn PyTorch Pandas NumPy Data AWS ETL Power BI Jupyter EDUCATION Mas

## 3. Upload the LLM

I selected Groq because it has a free tier API

In [ ]:
pip install groq

Here we import our private API key

In [ ]:
from google.colab import userdata
from groq import Groq
client = Groq(api_key=userdata.get('GROQ_API_KEY'))


In [ ]:
# This is to list all the models that are included in the Groq API
models = client.models.list()
for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
groq/compound
moonshotai/kimi-k2-instruct
whisper-large-v3
meta-llama/llama-prompt-guard-2-22m
moonshotai/kimi-k2-instruct-0905
qwen/qwen3-32b
whisper-large-v3-turbo
openai/gpt-oss-120b
llama-3.3-70b-versatile
openai/gpt-oss-20b
meta-llama/llama-4-scout-17b-16e-instruct
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
llama-3.1-8b-instant
groq/compound-mini
allam-2-7b


To implement the function that calls the LLM, we need to provide a JSON that requests the prompt and the system prompt.

In [ ]:
def question_llm(prompt,system):
    answer = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt},
            {"role": "system", "content": system}
        ]
    )
    return answer.choices[0].message.content

System will be temporary fixed here.

In [ ]:
system = """
Eres un asistente que gestiona las facturas y el CV del usuario. Responde la pregunta
basándote ÚNICAMENTE en el contexto proporcionado.

Si la información no está en el contexto, di "No tengo esa información".
"""

This function creates a prompt that includes the context before the question, to create the new prompt including the information from the PDFs.

In [ ]:
def context_prompt(question, context):
    prompt = f"""

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
"""
    return prompt

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search_relevant_chunks(question, vectorizer, vectors, chunks, top_k=min(3,len(chunks))):
    # Vectorize the question

    dummy_answer = question_llm(question, "Inicia tu respuesta escribiendo 'Pregunta: ', seguido de la pregunta escrita textualmente. Luego, escribe 'Respuesta:', y si no sabes la respuesta, provee una lista de posibles respuestas que sean cercanas a lo que pudiera estar preguntando.")
    vector_question = vectorizer.transform([dummy_answer])

    # Estimate Similarities
    sim = cosine_similarity(vector_question, vectors).flatten()

    # Find the indices of the top_k best matches
    # argsort sorts in ascending order, so we take the last 'top_k' elements
    best_indices = np.argsort(sim)[-top_k:][::-1]

    # Join the top chunks into a single context string
    relevant_chunks = [chunks[i] for i in best_indices]
    return "\n \n".join(relevant_chunks)

In [ ]:
def chatbot_pdf(question, vectorizer, vectors, chunks):
    context = search_relevant_chunks(question, vectorizer, vectors, chunks)
    prompt = context_prompt(question, context)
    answer = question_llm(prompt, system)
    return answer


In [ ]:
question = "De qué estaba trabajando cuando hice la compra?"
chatbot_pdf(question, vectorizer, vectors, chunks)

'Estabas trabajando como **AI Solutions Engineer**.'

Extra: Modifications for including context from the conversation.

In [ ]:
def preguntar_llm_avanzado(pregunta_usuario, instrucciones_sistema="Eres un asistente útil."):
    # We define the list with the 'system' instruction first
    # followed by the 'user' prompt.
    mensajes = [
        {"role": "system", "content": instrucciones_sistema},
        {"role": "user", "content": pregunta_usuario}
    ]

    respuesta = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=mensajes
    )
    return respuesta.choices[0].message.content

# Example of how it looks with chat history (Assistant role):
# history = [
#     {"role": "system", "content": "Eres un experto en PDF."},
#     {"role": "user", "content": "¿Qué es un PDF?"},
#     {"role": "assistant", "content": "Un PDF es un formato de documento..."},
#     {"role": "user", "content": "¿Y cómo lo edito?"}
# ]